In [1]:
print("Hello World")

Hello World


In [2]:
# !pip install cassandra-driver

In [3]:
import cassandra

In [4]:
print (cassandra.__version__)

3.29.2


In [5]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import json

# This secure connect bundle is autogenerated when you download your SCB, 
# if yours is different update the file name below
cloud_config= {
  'secure_connect_bundle': 'secure-connect-cassandra-demo.zip'
}

# This token JSON file is autogenerated when you download your token, 
# if yours is different update the file name below
with open("cassandra_demo-token.json") as f:
    secrets = json.load(f)

CLIENT_ID = secrets["clientId"]
CLIENT_SECRET = secrets["secret"]

auth_provider = PlainTextAuthProvider(CLIENT_ID, CLIENT_SECRET)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()

row = session.execute("select release_version from system.local").one()
if row:
  print(row[0])
else:
  print("An error occurred.")

4.0.11-5f20150d0dcb


In [6]:
row

Row(release_version='4.0.11-5f20150d0dcb')

### Command to use a keyspace

In [7]:
try:
    query="use employee_keyspace"
    session.execute(query)
    print("Inside the employee keyspace")
except Exception as err:
    print("Exception occured while using keyspace: ", err)

Inside the employee keyspace


### Command to create a table inside a KEyspace

In [8]:
try:
    query = """create table employee(
                emp_id int,
                emp_name varchar,
                emp_salary int,
                emp_dept varchar,
                emp_email varchar,
                emp_phone varchar,
                primary key (emp_id, emp_dept)
              )
            """
    session.execute(query)
    print("Table created inside the keyspace")
except Exception as err:
    print("Exception Occured while creating the table : ",err)

Table created inside the keyspace


### Alter the table in cassandra to drop a column

In [10]:
try:
    query = "alter table employee drop emp_email"
    session.execute(query)
    print("Column dropped successfully !!")
except Exception as err:
    print("Exception Occured while dropping the column: ",err)

Column dropped successfully !!


### Alter the table in cassandra to add a new column

In [11]:
try:
    query = "alter table employee add emp_email text"
    session.execute(query)
    print("Column added successfully !!")
except Exception as err:
    print("Exception Occured while adding the column: ",err)

Column added successfully !!


### Drop a table in cassandra

In [ ]:
# try:
#     query = "drop table employee"
#     session.execute(query)
#     print("Table dropped successfully !!")
# except Exception as err:
#     print("Exception Occured while dropping the table: ",err)

In [13]:
### Insert data into cassandra table

In [14]:
# 1
try:
    query = """insert into employee(emp_id, emp_name, emp_salary, emp_dept, emp_email, emp_phone) 
                values(1, 'Shashank', 10000, 'Software', 'abc.gmail.com','+91 768467474')"""
    session.execute(query)
    print("Record inserted successfully !!")
except Exception as err:
    print("Exception Occured while inserting the data into table: ",err)

Record inserted successfully !!


In [15]:
# 2
try:
    query = "insert into employee(emp_id, emp_name, emp_salary, emp_dept, emp_email, emp_phone) values(2, 'Rahul', 20000, 'IT', 'xyx.gmail.com','+91 908467474')"
    session.execute(query)
    print("Record inserted successfully !!")
except Exception as err:
    print("Exception Occured while inserting the data into table: ",err)

Record inserted successfully !!


In [16]:
# 3
try:
    query = "insert into employee(emp_id, emp_name, emp_salary, emp_dept, emp_email, emp_phone) values(3, 'Sunny', 22000, 'HR', 'klm.gmail.com','+91 800067474')"
    session.execute(query)
    print("Record inserted successfully !!")
except Exception as err:
    print("Exception Occured while inserting the data into table: ",err)

Record inserted successfully !!


In [17]:
# 4
try:
    query = "insert into employee(emp_id, emp_name, emp_salary, emp_dept, emp_email, emp_phone) values(4, 'Vishal', 30000, 'Software', 'mno.gmail.com','+91 600467474')"
    session.execute(query)
    print("Record inserted successfully !!")
except Exception as err:
    print("Exception Occured while inserting the data into table: ",err)

Record inserted successfully !!


### Select query on cassandra table

In [20]:
try:
    query = "select * from employee"
    result = session.execute(query)
    for row in result:
        print(row)
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Row(emp_id=1, emp_dept='Software', emp_email='abc.gmail.com', emp_name='Shashank', emp_phone='+91 768467474', emp_salary=10000)
Row(emp_id=2, emp_dept='IT', emp_email='xyx.gmail.com', emp_name='Rahul', emp_phone='+91 908467474', emp_salary=20000)
Row(emp_id=4, emp_dept='Software', emp_email='mno.gmail.com', emp_name='Vishal', emp_phone='+91 600467474', emp_salary=30000)
Row(emp_id=3, emp_dept='HR', emp_email='klm.gmail.com', emp_name='Sunny', emp_phone='+91 800067474', emp_salary=22000)


### Select query for specific columns in cassandra table and how to access from Row object

In [22]:
try:
    query = "select emp_id, emp_name from employee"
    result = session.execute(query)
    # option 1
    for row in result:
        print(f"Emp ID : {row[0]}, Emp Name : {row[1]}")
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Emp ID : 1, Emp Name : Shashank
Emp ID : 2, Emp Name : Rahul
Emp ID : 4, Emp Name : Vishal
Emp ID : 3, Emp Name : Sunny


### Write a query to get total count and max salary of employees

In [24]:
try:
    query = "select count(*) as row_count, max(emp_salary) as max_salary from employee"
    result = session.execute(query)
    row = result.one()
    print(row)
    #print(row[0])
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Row(row_count=4, max_salary=30000)


In [29]:
result.one()

Row(row_count=4, max_salary=30000)

### Write a query to filter data from cassandra table or how to use where clause
Rules for where clause - It can be used effectively with high performance for given below type of columns

1.) Partition Key (Single or Composite)\
                    OR\
2.) if Cluster column  used in where clause then it should be with Partition Key\
                   OR\
3.) A column on which we have applied the index\
                   OR\
4.) A column which is not part of partition key or index column or clustering column then we can use where clause but we have to
use keyword ALLOW FILTERING - it will be a super slow performance when data volume is very high

In [31]:
try:
    query = "select * from employee where emp_name='Shashank'"
    result = session.execute(query)
    row = result.one()
    print(row)
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Exception Occured while selecting the data from table:  Error from server: code=2200 [Invalid query] message="Cannot execute this query as it might involve data filtering and thus may have unpredictable performance. If you want to execute this query despite the performance unpredictability, use ALLOW FILTERING"


In [32]:
try:
    query = "select * from employee where emp_name='Shashank' ALLOW FILTERING"
    result = session.execute(query)
    row = result.one()
    print(row)
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Row(emp_id=1, emp_dept='Software', emp_email='abc.gmail.com', emp_name='Shashank', emp_phone='+91 768467474', emp_salary=10000)


In [33]:
# where clause for Partition key only or Rule no -1

try:
    query = "select * from employee where emp_id=2"
    result = session.execute(query)
    row = result.one()
    print(row)
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Row(emp_id=2, emp_dept='IT', emp_email='xyx.gmail.com', emp_name='Rahul', emp_phone='+91 908467474', emp_salary=20000)


In [34]:
# where clause for Clustering key only or Rule no - 2 

try:
    query = "select * from employee where emp_dept='Software' and emp_id=1"
    result = session.execute(query)
    row = result.one()
    print(row)
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Row(emp_id=1, emp_dept='Software', emp_email='abc.gmail.com', emp_name='Shashank', emp_phone='+91 768467474', emp_salary=10000)


### Group by in cassandra  -  Allowed for all columns which are part of Primary Key

In [35]:
# Follow given below rules
# Rule - 1 : Use only partition key in the group by
              #OR
# Rule - 2 : if Cluster key column is used then follow the actual declared sequence in the primary key
try:
    query = "select emp_id, sum(emp_salary) as sum_salary from employee group by emp_id"
    result = session.execute(query)
    row = result.one()
    for row in result:
        print("Emp ID : ", row[0])
        print("Sum Of Salary : ", row[1])
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Emp ID :  1
Sum Of Salary :  10000
Emp ID :  2
Sum Of Salary :  20000
Emp ID :  4
Sum Of Salary :  30000
Emp ID :  3
Sum Of Salary :  22000


In [37]:
# Group by in cassandra 
# Rule - 2 : if Cluster key column is used then follow the actual declared sequence in the primary key
try:
    query = "select emp_id, emp_dept, sum(emp_salary) as sum_salary from employee group by emp_dept"
    result = session.execute(query)
    row = result.one()
    for row in result:
        print("Emp ID : ", row[0])
        print("Emp Dept : ", row[1])
        print("Sum Of Salary : ", row[2])
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Exception Occured while selecting the data from table:  Error from server: code=2200 [Invalid query] message="Group by currently only support groups of columns following their declared order in the PRIMARY KEY"


In [49]:
# Group by in cassandra 
# Rule - 2 : if Cluster key column is used then follow the actual declared sequence in the primary key
try:
    query = "select emp_id, emp_dept, sum(emp_salary) as sum_salary from employee group by emp_id, emp_dept"
    result = session.execute(query)
    row = result.one()
    for row in result:
        print("Emp ID : ", row[0])
        print("Emp Dept : ", row[1])
        print("Sum Of Salary : ", row[2])
except Exception as err:
    print("Exception Occured while selecting the data from table: ",err)

Emp ID :  1
Emp Dept :  Software
Sum Of Salary :  10000
Emp ID :  2
Emp Dept :  IT
Sum Of Salary :  20000
Emp ID :  4
Emp Dept :  Software
Sum Of Salary :  30000
Emp ID :  3
Emp Dept :  HR
Sum Of Salary :  22000


In [54]:
import pandas as pd

In [55]:
query = "select emp_id, emp_dept, sum(emp_salary) as sum_salary from employee group by emp_id, emp_dept"
result = session.execute(query)
df = pd.DataFrame(result.all())
df

,emp_id,emp_dept,sum_salary
0,1,Software,10000
1,2,IT,20000
2,4,Software,30000
3,3,HR,22000


In [52]:
result.all()

[]